In [0]:
CATALOG = "energy_puls"
LANDING = f"/Volumes/{CATALOG}/landing/landing"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"
from pyspark.sql import functions as F


(spark.read.option("header","true").option("inferSchema","true")
   .option("recursiveFileLookup","true")
   .csv(f"{LANDING}/enedis/annual/")
   .withColumn("_ingested_at", F.current_timestamp())
   .write.mode("overwrite").saveAsTable(f"{BRONZE}.enedis_raw"))


meteo = (spark.read.option("multiline","true")
   .option("recursiveFileLookup","true")
   .json(f"{LANDING}/meteo/daily/"))
(meteo.select("region_code","region_name","date",
        F.explode(F.arrays_zip("hourly.time","hourly.temperature_2m")).alias("h"))
   .select("region_code","region_name","date",
           F.col("h.time").alias("heure"),
           F.col("h.temperature_2m").alias("temperature"))
   .write.mode("overwrite").saveAsTable(f"{SILVER}.meteo_clean"))


(spark.read.parquet(f"{LANDING}/insee/communes_reference.parquet")
   .write.mode("overwrite").saveAsTable(f"{SILVER}.communes"))

for t in ["bronze.enedis_raw","silver.meteo_clean","silver.communes"]:
    print(t, spark.table(f"{CATALOG}.{t}").count())